# Phase 2 analysis: Peptides-func re-baseline

Reads the run folders written by the Phase 2 sweep (`configs/sweeps/phase2.yaml`) and shows
every table and figure from `gnn_mech.analysis`: test AP vs published numbers, AP stratified by
effective resistance and graph size, Jacobian slopes with graph-clustered bootstrap CIs and the
slope difference vs the transformer (alpha = 1), entropy vs size per head, and the range measure.
All bootstraps are seeded (`SEED`), so reruns give identical numbers.

Set `OUT_DIR` to the sweep's `out_dir` (on Colab, mount Drive first; on CPU this runs as is).

In [ ]:
import os
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/message-passing-in-gnns
    OUT_DIR = '/content/drive/MyDrive/Thesis/gnn_mech/runs'
    REPORT_DIR = '/content/drive/MyDrive/Thesis/gnn_mech/reports/phase2'
else:
    OUT_DIR = './runs'
    REPORT_DIR = './reports/phase2'
EXPS = ['peptides_func_alpha', 'peptides_func_gcn']
N_BOOT, SEED = 1000, 0

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from gnn_mech import analysis as A

plt.rcParams.update(A.RC_PARAMS)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
res = A.run_analysis(OUT_DIR, REPORT_DIR, EXPS, n_boot=N_BOOT, seed=SEED)
res['runs'][['exp', 'model', 'seed', 'test', 'best_epoch', 'epochs_run', 'n_params', 'has_preds', 'has_measurements']]

## Performance (gate G1)

G1: alpha = 0.5 mean test AP >= 0.63, alpha = 1 >= 0.60, GCN >= 0.66 (or the gap documented).

In [ ]:
display(res['performance'])
display(Image(f'{REPORT_DIR}/figures/ap_bars.png'))

## AP stratified by effective resistance and graph size (tertiles)

In [ ]:
display(res['stratified_ap'])
display(Image(f'{REPORT_DIR}/figures/stratified_ap.png'))

## Jacobian sensitivity (gate G2)

G2: the slope difference vs the transformer (alpha = 1) has a bootstrap CI excluding 0, or we
report that it does not. Peptides graphs are near-trees, so resistance and hops are nearly
collinear (see `collinearity`).

In [ ]:
display(res['jacobian_slopes'])
display(res['jacobian_differences'])
print(res['collinearity'])
display(Image(f'{REPORT_DIR}/figures/jacobian_vs_resistance.png'))

## Attention entropy vs graph size, per layer and head

In [ ]:
display(res['entropy'])
display(Image(f'{REPORT_DIR}/figures/entropy_vs_size.png'))

## Range (Bamberger et al. 2025)

Node level: L1-normalized Jacobian range in hops and effective resistance, one value per measured
target node. Graph level: from Hessians of the pooled logits; undefined (all-zero Hessian) for
piecewise-linear models such as alpha = 0 and the GCN.

In [ ]:
display(res['range_node'])
display(res['range_graph'])
print(res['range_reading'])
display(Image(f'{REPORT_DIR}/figures/range_distributions.png'))